# Load DBSQL Rates and Warehouse Configuration

This notebook creates two tables:

1. **`dbsql_rates`** - Fixed DBU/hour for all warehouse types (Classic, Pro, Serverless)
2. **`dbsql_warehouse_config`** - VM configuration for Classic/Pro (driver/worker instance types)

### Cost Calculation

| Type | Formula |
|------|---------|  
| **Serverless** | `DBU/hr × DBU_price × hours` |
| **Classic/Pro** | `DBU/hr × DBU_price × hours` + `(driver_vm + workers × worker_vm) × hours` |

**Source:** Databricks Documentation


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_DBSQL_RATES = f"{CATALOG}.{SCHEMA}.dbsql_rates"
TABLE_DBSQL_WAREHOUSE_CONFIG = f"{CATALOG}.{SCHEMA}.dbsql_warehouse_config"

print(f"✅ Config: {CATALOG}.{SCHEMA}")
print(f"   Tables: {TABLE_DBSQL_RATES}, {TABLE_DBSQL_WAREHOUSE_CONFIG}")


In [ ]:
import pandas as pd
from datetime import datetime

# =============================================================================
# DBSQL RATES - Fixed DBU/hour per warehouse size (with cloud column for future flexibility)
# Source: https://docs.databricks.com/aws/en/resources/pricing#sql-serverless-sku
#         https://docs.databricks.com/gcp/en/resources/pricing#sql-serverless-sku
#         https://learn.microsoft.com/en-gb/azure/databricks/resources/pricing#sql-serverless-sku
#
# NOTE: DBU/hour rates are currently THE SAME across clouds, but we include
#       cloud column for future flexibility (in case rates diverge).
#       The difference is:
#       - Classic/Pro: Customer pays DBU + VM costs separately
#       - Serverless: DBU price includes compute (no separate VM cost)
# =============================================================================

# Clouds
CLOUDS = ["AWS", "AZURE", "GCP"]

# Warehouse sizes (T-shirt sizing)
WAREHOUSE_SIZES = [
    "2X-Small", "X-Small", "Small", "Medium", "Large", 
    "X-Large", "2X-Large", "3X-Large", "4X-Large"
]

# DBU/hour rates - currently SAME for all clouds (from official docs)
DBU_PER_HOUR = {
    "2X-Small": 4,
    "X-Small": 6,
    "Small": 12,
    "Medium": 24,
    "Large": 40,
    "X-Large": 80,
    "2X-Large": 144,
    "3X-Large": 272,
    "4X-Large": 528,
}

# Mapping to dbu_prices.product_type for joining
WAREHOUSE_TYPE_TO_SKU = {
    "classic": "SQL_COMPUTE",
    "pro": "SQL_PRO_COMPUTE",
    "serverless": "SERVERLESS_SQL_COMPUTE",
}

# Generate rates for all clouds × warehouse types × sizes
dbsql_rates_data = []
for cloud in CLOUDS:
    for warehouse_type in ["classic", "pro", "serverless"]:
        for warehouse_size, dbu_per_hour in DBU_PER_HOUR.items():
            dbsql_rates_data.append({
                "cloud": cloud,
                "warehouse_type": warehouse_type,
                "warehouse_size": warehouse_size,
                "sku_product_type": WAREHOUSE_TYPE_TO_SKU[warehouse_type],  # For joining to dbu_prices
                "dbu_per_hour": float(dbu_per_hour),
                "includes_compute": warehouse_type == "serverless",  # True for serverless only
                "source": "Databricks Documentation",
                "is_active": True,
                "updated_at": datetime.utcnow()
            })

df_dbsql_rates = pd.DataFrame(dbsql_rates_data)
print(f"✅ Created {len(df_dbsql_rates)} DBSQL rate entries")
print(f"   Clouds: {df_dbsql_rates['cloud'].unique().tolist()}")
print(f"   Types: {df_dbsql_rates['warehouse_type'].unique().tolist()}")
print(f"   Sizes: {df_dbsql_rates['warehouse_size'].unique().tolist()}")
print(f"   Total: {len(CLOUDS)} clouds × 3 types × {len(WAREHOUSE_SIZES)} sizes = {len(df_dbsql_rates)} rows")

# Display pivot table for easy viewing (show AWS as example, all clouds have same rates)
pivot = df_dbsql_rates[df_dbsql_rates['cloud'] == 'AWS'].pivot(
    index='warehouse_size', columns='warehouse_type', values='dbu_per_hour')
pivot = pivot.reindex(WAREHOUSE_SIZES)  # Reorder by size
print("\n📊 DBU/hour by Warehouse Type and Size (same for all clouds):")
print(pivot)


In [ ]:
# =============================================================================
# DBSQL WAREHOUSE CONFIG - VM Configuration for Classic/Pro
# (Serverless does NOT have user-configurable VMs)
# 
# Sources:
# - AWS: https://docs.databricks.com/aws/en/compute/sql-warehouse/warehouse-behavior#sizing-and-cluster-provisioning
# - GCP: https://docs.databricks.com/gcp/en/compute/sql-warehouse/warehouse-behavior#sizing-and-cluster-provisioning
# - Azure: https://learn.microsoft.com/en-gb/azure/databricks/compute/sql-warehouse/warehouse-behavior#sizing-and-cluster-provisioning
# =============================================================================

# AWS instance types for SQL Warehouses (from official docs)
# All workers use i3.2xlarge, driver varies by size
AWS_INSTANCES = {
    #             (driver_instance, worker_instance)
    "2X-Small":  ("i3.2xlarge",  "i3.2xlarge"),
    "X-Small":   ("i3.2xlarge",  "i3.2xlarge"),
    "Small":     ("i3.4xlarge",  "i3.2xlarge"),
    "Medium":    ("i3.8xlarge",  "i3.2xlarge"),
    "Large":     ("i3.8xlarge",  "i3.2xlarge"),
    "X-Large":   ("i3.16xlarge", "i3.2xlarge"),
    "2X-Large":  ("i3.16xlarge", "i3.2xlarge"),
    "3X-Large":  ("i3.16xlarge", "i3.2xlarge"),
    "4X-Large":  ("i3.16xlarge", "i3.2xlarge"),
}

# Azure instance types for SQL Warehouses (from official docs)
# All workers use Standard_E8ds_v4, driver varies by size
AZURE_INSTANCES = {
    #             (driver_instance, worker_instance)
    "2X-Small":  ("Standard_E8ds_v4",  "Standard_E8ds_v4"),
    "X-Small":   ("Standard_E8ds_v4",  "Standard_E8ds_v4"),
    "Small":     ("Standard_E16ds_v4", "Standard_E8ds_v4"),
    "Medium":    ("Standard_E32ds_v4", "Standard_E8ds_v4"),
    "Large":     ("Standard_E32ds_v4", "Standard_E8ds_v4"),
    "X-Large":   ("Standard_E64ds_v4", "Standard_E8ds_v4"),
    "2X-Large":  ("Standard_E64ds_v4", "Standard_E8ds_v4"),
    "3X-Large":  ("Standard_E64ds_v4", "Standard_E8ds_v4"),
    "4X-Large":  ("Standard_E64ds_v4", "Standard_E8ds_v4"),
}

# GCP instance types for SQL Warehouses (from official docs)
# All workers use n2-highmem-8, driver varies by size
GCP_INSTANCES = {
    #             (driver_instance, worker_instance)
    "2X-Small":  ("n2-highmem-8",  "n2-highmem-8"),
    "X-Small":   ("n2-highmem-8",  "n2-highmem-8"),
    "Small":     ("n2-highmem-16", "n2-highmem-8"),
    "Medium":    ("n2-highmem-32", "n2-highmem-8"),
    "Large":     ("n2-highmem-32", "n2-highmem-8"),
    "X-Large":   ("n2-highmem-64", "n2-highmem-8"),
    "2X-Large":  ("n2-highmem-64", "n2-highmem-8"),
    "3X-Large":  ("n2-highmem-64", "n2-highmem-8"),
    "4X-Large":  ("n2-highmem-64", "n2-highmem-8"),
}

# Warehouse size to FIXED worker count (from official docs)
# Note: Even 2X-Small has 1 worker!
WAREHOUSE_CLUSTER_CONFIG = {
    #               (driver_count, worker_count)
    "2X-Small":  (1, 1),       # 1 driver + 1 worker
    "X-Small":   (1, 2),       # 1 driver + 2 workers
    "Small":     (1, 4),       # 1 driver + 4 workers
    "Medium":    (1, 8),       # 1 driver + 8 workers
    "Large":     (1, 16),      # 1 driver + 16 workers
    "X-Large":   (1, 32),      # 1 driver + 32 workers
    "2X-Large":  (1, 64),      # 1 driver + 64 workers
    "3X-Large":  (1, 128),     # 1 driver + 128 workers
    "4X-Large":  (1, 256),     # 1 driver + 256 workers
}

# Generate warehouse config entries
warehouse_config_data = []

CLOUD_INSTANCES = {
    "AWS": AWS_INSTANCES,
    "AZURE": AZURE_INSTANCES,
    "GCP": GCP_INSTANCES,
}

for cloud, instances in CLOUD_INSTANCES.items():
    for warehouse_type in ["classic", "pro"]:  # Only Classic and Pro have VMs
        for warehouse_size, (driver_count, worker_count) in WAREHOUSE_CLUSTER_CONFIG.items():
            driver_instance, worker_instance = instances[warehouse_size]
            
            warehouse_config_data.append({
                "cloud": cloud,
                "warehouse_type": warehouse_type,
                "warehouse_size": warehouse_size,
                "driver_count": driver_count,
                "driver_instance_type": driver_instance,
                "worker_count": worker_count,
                "worker_instance_type": worker_instance,  # All sizes have workers
                "source": "Databricks Documentation",
                "is_active": True,
                "updated_at": datetime.utcnow()
            })

df_warehouse_config = pd.DataFrame(warehouse_config_data)
print(f"✅ Created {len(df_warehouse_config)} warehouse config entries")
print(f"   Clouds: {df_warehouse_config['cloud'].unique().tolist()}")
print(f"   Types: {df_warehouse_config['warehouse_type'].unique().tolist()}")

# Show sample
print("\n📊 Sample AWS Classic configs:")
print(df_warehouse_config[(df_warehouse_config['cloud'] == 'AWS') & (df_warehouse_config['warehouse_type'] == 'classic')][['warehouse_size', 'driver_count', 'driver_instance_type', 'worker_count', 'worker_instance_type']].to_string(index=False))


In [ ]:
# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema {CATALOG}.{SCHEMA} ready")


In [ ]:
# Save DBSQL Rates table
spark_df_rates = spark.createDataFrame(df_dbsql_rates)

spark_df_rates.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_DBSQL_RATES)

print(f"✅ Saved {len(df_dbsql_rates)} rows to {TABLE_DBSQL_RATES}")


In [ ]:
# Save Warehouse Config table
spark_df_config = spark.createDataFrame(df_warehouse_config)

spark_df_config.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_DBSQL_WAREHOUSE_CONFIG)

print(f"✅ Saved {len(df_warehouse_config)} rows to {TABLE_DBSQL_WAREHOUSE_CONFIG}")


In [ ]:
# =============================================================================
# VERIFY: Display the saved data
# =============================================================================

print("📊 DBSQL Rates Table:")
display(spark.sql(f"""
    SELECT warehouse_type, warehouse_size, dbu_per_hour, includes_compute
    FROM {TABLE_DBSQL_RATES}
    ORDER BY 
        CASE warehouse_type 
            WHEN 'classic' THEN 1 
            WHEN 'pro' THEN 2 
            WHEN 'serverless' THEN 3 
        END,
        CASE warehouse_size
            WHEN '2X-Small' THEN 1
            WHEN 'X-Small' THEN 2
            WHEN 'Small' THEN 3
            WHEN 'Medium' THEN 4
            WHEN 'Large' THEN 5
            WHEN 'X-Large' THEN 6
            WHEN '2X-Large' THEN 7
            WHEN '3X-Large' THEN 8
            WHEN '4X-Large' THEN 9
        END
"""))


In [ ]:
print("📊 DBSQL Warehouse Config Table (AWS only for brevity):")
display(spark.sql(f"""
    SELECT cloud, warehouse_type, warehouse_size, 
           driver_count, driver_instance_type,
           worker_count, worker_instance_type
    FROM {TABLE_DBSQL_WAREHOUSE_CONFIG}
    WHERE cloud = 'AWS'
    ORDER BY 
        warehouse_type,
        CASE warehouse_size
            WHEN '2X-Small' THEN 1
            WHEN 'X-Small' THEN 2
            WHEN 'Small' THEN 3
            WHEN 'Medium' THEN 4
            WHEN 'Large' THEN 5
            WHEN 'X-Large' THEN 6
            WHEN '2X-Large' THEN 7
            WHEN '3X-Large' THEN 8
            WHEN '4X-Large' THEN 9
        END
"""))


In [ ]:
# =============================================================================
# EXAMPLE: Cost Calculation Query
# =============================================================================

print("""
📝 EXAMPLE: Calculate hourly cost for a Small SQL Warehouse on AWS

Components:
1. DBU Cost = dbu_per_hour × dbu_price
2. VM Cost = (driver_count × driver_vm_cost) + (worker_count × worker_vm_cost)
3. Total = DBU Cost + VM Cost (only for Classic/Pro)
""")

# Example calculation - compare all 3 types for a Small warehouse
display(spark.sql(f"""
    WITH warehouse_details AS (
        SELECT 
            r.warehouse_type,
            r.warehouse_size,
            r.dbu_per_hour,
            r.includes_compute,
            c.cloud,
            c.driver_count,
            c.driver_instance_type,
            c.worker_count,
            c.worker_instance_type
        FROM {TABLE_DBSQL_RATES} r
        LEFT JOIN {TABLE_DBSQL_WAREHOUSE_CONFIG} c
            ON r.warehouse_type = c.warehouse_type
            AND r.warehouse_size = c.warehouse_size
        WHERE r.warehouse_size = 'Small'
          AND (c.cloud = 'AWS' OR c.cloud IS NULL)
    )
    SELECT 
        warehouse_type,
        warehouse_size,
        cloud,
        dbu_per_hour,
        includes_compute,
        driver_count,
        driver_instance_type,
        worker_count,
        worker_instance_type,
        CASE 
            WHEN includes_compute THEN 'DBU only (compute included)'
            ELSE 'DBU + VM cost'
        END as cost_components
    FROM warehouse_details
    ORDER BY warehouse_type
"""))


In [ ]:
# =============================================================================
# FULL COST CALCULATION with VM Costs
# =============================================================================

print("📊 Full Cost Calculation Example (Small warehouse, AWS, us-east-1):")
print("   Assuming DBU price = $0.22/DBU for SQL Pro, $0.15/DBU for SQL Classic")

display(spark.sql(f"""
    WITH warehouse_details AS (
        SELECT 
            r.warehouse_type,
            r.warehouse_size,
            r.dbu_per_hour,
            r.includes_compute,
            c.cloud,
            c.driver_count,
            c.driver_instance_type,
            c.worker_count,
            c.worker_instance_type
        FROM {TABLE_DBSQL_RATES} r
        LEFT JOIN {TABLE_DBSQL_WAREHOUSE_CONFIG} c
            ON r.warehouse_type = c.warehouse_type
            AND r.warehouse_size = c.warehouse_size
        WHERE r.warehouse_size = 'Small'
          AND (c.cloud = 'AWS' OR c.cloud IS NULL)
    ),
    with_vm_costs AS (
        SELECT 
            w.*,
            v.cost_per_hour as driver_vm_cost,
            v2.cost_per_hour as worker_vm_cost
        FROM warehouse_details w
        LEFT JOIN {CATALOG}.{SCHEMA}.vm_costs v
            ON w.driver_instance_type = v.instance_type
            AND w.cloud = v.cloud
            AND v.pricing_tier = 'on_demand'
            AND v.region = 'us-east-1'
        LEFT JOIN {CATALOG}.{SCHEMA}.vm_costs v2
            ON w.worker_instance_type = v2.instance_type
            AND w.cloud = v2.cloud
            AND v2.pricing_tier = 'on_demand'
            AND v2.region = 'us-east-1'
    )
    SELECT 
        warehouse_type,
        warehouse_size,
        cloud,
        dbu_per_hour,
        -- DBU cost calculation
        CASE 
            WHEN warehouse_type = 'classic' THEN ROUND(dbu_per_hour * 0.15, 2)
            ELSE ROUND(dbu_per_hour * 0.22, 2)
        END as dbu_cost_per_hour,
        -- VM details
        driver_vm_cost,
        worker_count,
        worker_vm_cost,
        -- Total VM cost
        ROUND(COALESCE(driver_vm_cost, 0) + COALESCE(worker_count * worker_vm_cost, 0), 2) as total_vm_cost_per_hour,
        -- Total cost
        CASE 
            WHEN includes_compute THEN 
                ROUND(dbu_per_hour * 0.22, 2)
            WHEN warehouse_type = 'classic' THEN 
                ROUND(dbu_per_hour * 0.15 + COALESCE(driver_vm_cost, 0) + COALESCE(worker_count * worker_vm_cost, 0), 2)
            ELSE 
                ROUND(dbu_per_hour * 0.22 + COALESCE(driver_vm_cost, 0) + COALESCE(worker_count * worker_vm_cost, 0), 2)
        END as total_cost_per_hour
    FROM with_vm_costs
    ORDER BY warehouse_type
"""))


## Summary

### Tables Created

| Table | Rows | Description |
|-------|------|-------------|
| `dbsql_rates` | 81 | Fixed DBU/hour for Classic, Pro, Serverless (3 clouds × 3 types × 9 sizes) |
| `dbsql_warehouse_config` | 54 | VM config for Classic/Pro only (3 clouds × 2 types × 9 sizes) |

### Table Schemas

**`dbsql_rates`**:
| Column | Description |
|--------|-------------|
| cloud | AWS, AZURE, GCP |
| warehouse_type | classic, pro, serverless |
| warehouse_size | 2X-Small to 4X-Large |
| **sku_product_type** | **SQL_COMPUTE, SQL_PRO_COMPUTE, SERVERLESS_SQL_COMPUTE** (for joining to dbu_prices) |
| dbu_per_hour | Fixed DBU consumption |
| includes_compute | TRUE only for serverless |

**`dbsql_warehouse_config`**:
| Column | Description |
|--------|-------------|
| cloud | AWS, AZURE, GCP |
| warehouse_type | classic, pro |
| warehouse_size | 2X-Small to 4X-Large |
| driver_count | Number of drivers (always 1) |
| driver_instance_type | e.g., i3.xlarge |
| worker_count | Fixed number of workers |
| worker_instance_type | e.g., i3.xlarge |

### Cost Formula

**Serverless:**
```
hourly_cost = dbu_per_hour × dbu_price
```

**Classic/Pro:**
```
dbu_cost = dbu_per_hour × dbu_price
vm_cost = (driver_count × driver_vm_cost) + (worker_count × worker_vm_cost)
hourly_cost = dbu_cost + vm_cost
```

### Key Differences

| Type | DBU Rate | VM Cost | Use Case |
|------|----------|---------|----------|
| **Classic** | Same (e.g., 12 DBU/hr for Small) | Customer pays | Cost optimization, full control |
| **Pro** | Same (e.g., 12 DBU/hr for Small) | Customer pays | Advanced features + cost control |
| **Serverless** | Same (e.g., 12 DBU/hr for Small) | Included in DBU | Fully managed, no VM overhead |

**Note:** DBU/hour rates are identical across all warehouse types. The difference is in billing:
- **Classic/Pro**: You pay DBU cost + VM cost (from cloud provider)
- **Serverless**: You only pay DBU cost (compute is included in the DBU price)

### Notes

- VM costs come from `vm_costs` table (on_demand, spot, reserved)
- Driver/worker instance types from `dbsql_warehouse_config` table
- DBU prices vary by cloud and region (from `system.billing.list_prices`)
